# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Emeka-techDev/ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item for one client on one reporting date; I will use March 2026 as my development window.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {hf_token})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

daily = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"


con.sql("""
SELECT
    month,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY month;
""")


con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1;
""")




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

March 2026 contains 9,841,378 rows in the daily performance table. Also, there are no duplicate combinations of client_hash_id + content_hash_id + report_date in March 2026.


┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

gsc_impressions

* Available when? Known at the decision moment because it comes from the historical GSC performance data available for the March reporting period.

gsc_clicks

* Available when? Known at the decision moment because it is historical GSC performance for the content.

gsc_avg_position

* Available when? Known at the decision moment because it describes the content's historical search position.

ga4_sessions

* Available when? Known at the decision moment because it is historical GA4 traffic data for the content.

scroll_events

* Available when? Known at the decision moment because it measures historical user interaction with the content.


**Label/Proxy**

*  future performance change (proxy)

Proxy label: future content performance change, since the warehouse does not directly contain a refresh-success outcome.

**Context**
* client_hash_id
* content_hash_id
* report_date
* month

**excluded**
* future-period metrics
* June 2026 data
* content_hash_id
* client_hash_id


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — Grain**

In [ ]:
con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

**Query 2 — Count + date span**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03';
""")

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

**Query 3 — GSC availability**

In [ ]:
con.sql("""
SELECT
    COUNT(*) AS gsc_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ gsc_available_rows │
│       int64        │
├────────────────────┤
│            3611061 │
└────────────────────┘

# **Test Leakage**

In [2]:
con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
)

SELECT *
FROM march
LIMIT 10;
""")

NameError: name 'con' is not defined

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation — no direct refresh-success outcome: The warehouse can show content performance and characteristics, but it does not tell us whether refreshing a piece of content actually caused its future performance to improve. Therefore, future performance change is only a proxy for refresh opportunity, not a true measure of refresh success.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.